In [1]:
import os
from langchain_core.documents import Document

In [2]:
os.makedirs("data/text_files", exist_ok = True)
os.makedirs("data/pdfs", exist_ok = True)

In [3]:
from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader, TextLoader, CSVLoader

# pdfDoc = PyMuPDFLoader('data/pdfs/Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf')
# docs = pdfDoc.load()
# len(docs)

doc = DirectoryLoader(
    "data/pdfs",
    loader_cls= PyMuPDFLoader,
    glob = "**/*.pdf"
)

documents = doc.load()
documents

d:\DataScienceProjects\Langchain-RAG-Optimized\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0}, page_content='M A N N I N G\nBen Wilson\nIN ACTION'),
 Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF E

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_pdf(directory):
    all_documents = []
    dir = Path(directory)

    # pdf_files = list(pdf_dir.glob('**/*.pdf')) 
    files = list(dir.glob('**/*.*')) 

    print(f'Found {len(files)} files in the directory.')

    loader_map = {
        '**/*.pdf' : PyMuPDFLoader,
        '**/*.txt' : TextLoader,
        '**/*.csv' : CSVLoader
    }

    # for pdf_file in pdf_files:
    #     # loader = PyMuPDFLoader(pdf_file)
    #     loader = DirectoryLoader(
    #         pdf_files
    #     )
    #     documents = loader.load()

    for pattern, loader_cls in loader_map.items():
        loader = DirectoryLoader(
            directory,
            glob=pattern,
            loader_cls = loader_cls,
            use_multithreading=True,
            max_concurrency=4,
            silent_errors=True # Skips corrupted files instead of reading.
        )

        all_documents.extend(loader.load())

    for doc in all_documents:
        # path = doc.metadata.get('source', 'unknown')
        # print(path.split('\\')[-1][-3:])
        # doc.metadata['source'] = path.split('\\')[-1]
        # doc.metadata["file_type"] = 'pdf'
        p = Path(doc.metadata['source'])
        # path_str = doc.metadata.get("source", "")
        # path_obj = Path(path_str)
        
        # .suffix gives you '.pdf' (includes the dot)
        # .stem gives you 'filename' (no extension)
        doc.metadata["filetype"] = p.suffix[1:].lower()
        doc.metadata["filename"] = p.name
    
    # all_documents.extend(documents)
    print(f'Total Documents Loaded: {len(documents)}')

    return all_documents

all_pdfs_documents = process_all_pdf('./data')

Found 15 files in the directory.
Total Documents Loaded: 2307


In [5]:
import tiktoken
# Create chunks.
# 1. Initialize the tokenizer for your specific model
# 'cl100k_base' is used for GPT-3.5, GPT-4, and GPT-4o
tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Define a function that takes text and returns the token count
def tiktoken_len(text):
    tokens = tokenizer.encode(text, disallowed_special=())
    return len(tokens)

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_doc = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_doc)} chunks.")

    #Show example of chuncks.
    if split_doc:
        print(f'\nExample chunks:')
        print(f'Content: {split_doc[0].page_content[:200]}...')
        print(f'Metadata: {split_doc[0].metadata}')

    return split_doc

In [6]:
chunks = split_documents(all_pdfs_documents)
chunks

Split 2307 documents into 5703 chunks.

Example chunks:
Content: M A N N I N G
Ben Wilson
IN ACTION...
Metadata: {'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0, 'filetype': 'pdf', 'filename': 'Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)', 'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-06T16:40:24+00:00', 'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'file_path': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf', 'total_pages': 578, 'format': 'PDF 1.6', 'title': 'Machine Learning Engineering in Action', 'author': 'Ben Wilson', 'subject': '', 'keywords': '', 'moddate': 'D:20220322233007', 'trapped': '', 'modDate': 'D:20220322233007', 'creationDate': 'D:20220306164024Z', 'page': 0, 'filetype': 'pdf', 'filename': 'Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf'}, page_content='M A N N I N G\nBen Wilson\nIN ACTION'),
 Document(metadata={'producer': 'Acrobat Distiller 20.0 (Wind

# Embedding and Vector Store DB

In [7]:
    import numpy as np
    from sentence_transformers import SentenceTransformer
    import chromadb
    from chromadb.config import Settings
    import uuid
    from typing import List, Dict, Any, Tuple
    from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        # Use huggingface model name for sentence embedding
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model is loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except: 
            print(f'Error loading model {self.model_name}: {e}')
            raise
    
    def generate_embeddings(self, documents):
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(documents)} docuemnts")
        embeddings = self.model.encode(documents, show_progress_bar=True)
        return embeddings

    # LangChain/Ragas specifically looks for this method name
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()

    # And this one for single queries
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model is loaded successfully. Embedding dimension: 384


## VectorDB

In [9]:
class VectorStore:
    def __init__(self, collection_name = "pdf_documents", persistent_directory = "./data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        # Create persistent ChromaDB client
        os.makedirs(self.persistent_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=self.persistent_directory, 
            settings=Settings(allow_reset=True, anonymized_telemetry=False)
            )

        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                'hnsw:space': 'cosine',
                'description': 'PDF document embeddings for RAG'
                }
        )

        print(f"Vector store initialized. Collection: {self.collection_name}")
        print(f"Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
    
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        # Batching configuration (Optimal for local SSDs)
        batch_size = 100 
        total_docs = len(documents)
        
        for i in range(0, total_docs, batch_size):
            batch_docs = documents[i:i + batch_size]
            batch_embeddings = embeddings[i:i + batch_size]

            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(batch_docs, batch_embeddings)):
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document conntent
                documents_text.append(doc.page_content)

                # Embeddings
                embeddings_list.append(embedding.tolist())
            
            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    metadatas=metadatas,
                    documents=documents_text,
                    embeddings=embeddings_list
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
            except Exception as e:
                print(f"Error adding docuemnts to vector store: {e}")
                raise


vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 55343


In [10]:
import gc

def ingest_data(vector_store, embedding_manager, chunks):
    # 1. Convert to embeddings
    texts = [doc.page_content for doc in chunks]
    embeddings = embedding_manager.generate_embeddings(texts)

    # 2. Store into the database
    vector_store.add_documents(chunks, embeddings)
    
    # 3. Explicitly dereference the heavy lists
    del texts
    del embeddings
    # 'chunks' is still available if needed outside this function
    
    # 4. Optional: Suggest to Python that now is a good time to clean up   
    gc.collect()

# Call the function
ingest_data(vector_store, embedding_manager, chunks)

Generating embeddings for 5703 docuemnts


Batches: 100%|██████████| 179/179 [02:45<00:00,  1.08it/s]


Adding 5703 documents to vector store...
Successfully added 5703 documents to vector store
Total documents in collection: 55443
Successfully added 5703 documents to vector store
Total documents in collection: 55543
Successfully added 5703 documents to vector store
Total documents in collection: 55643
Successfully added 5703 documents to vector store
Total documents in collection: 55743
Successfully added 5703 documents to vector store
Total documents in collection: 55843
Successfully added 5703 documents to vector store
Total documents in collection: 55943
Successfully added 5703 documents to vector store
Total documents in collection: 56043
Successfully added 5703 documents to vector store
Total documents in collection: 56143
Successfully added 5703 documents to vector store
Total documents in collection: 56243
Successfully added 5703 documents to vector store
Total documents in collection: 56343
Successfully added 5703 documents to vector store
Total documents in collection: 56443
Su

In [11]:
# vector_store.client.reset() # Wipes all collections and data

In [12]:
# # vector_store.client.reset() # Wipes all collections and data
# # Convert the text to embeddings
# texts = [doc.page_content for doc in chunks]

# #Generate Embedding
# embeddings = embedding_manager.generate_embeddings(texts)

# #store into the vector database
# vector_store.add_documents(chunks, embeddings)


In [13]:
class RAGRetrievar:
    """Handles query based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing the documents
            embedding manager: Manager for generating query embeddings
        
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve (self, query: str, top_k: int = 5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimun similarity score threshold

        Returns:
            List of dictionaries contaning retrived documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # print('Second query embeddding', query_embedding)

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate (zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distane)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            
            else:
                print("No documents found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f'Error during retrieval: {e}')
            raise []
        
rag_retriever = RAGRetrievar(vector_store, embedding_manager)

In [14]:
rag_retriever.retrieve("What is Computational complexity?")

Retrieving documents for query: 'What is Computational complexity?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.22it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_479c736a_36',
  'content': 'shorthand as Big O.\nA.1.1\nA gentle introduction to complexity\nComputational complexity is, at its heart, a worst-case estimation of how long it will take\nfor a computer to work through an algorithm. Space complexity, on the other hand, is',
  'metadata': {'subject': '',
   'moddate': 'D:20220322233007',
   'doc_index': 36,
   'producer': 'Acrobat Distiller 20.0 (Windows); modified using iText® 7.1.15 ©2000-2021 iText Group NV (AGPL-version)',
   'total_pages': 578,
   'page': 536,
   'content_length': 238,
   'trapped': '',
   'creator': 'FrameMaker 16.0.1(Foxit Advanced PDF Editor)',
   'source': 'data\\pdfs\\Ben Wilson - Machine Learning Engineering in Action-Manning Publications (2022)(Z-Lib.io).pdf',
   'creationDate': 'D:20220306164024Z',
   'creationdate': '2022-03-06T16:40:24+00:00',
   'format': 'PDF 1.6',
   'author': 'Ben Wilson',
   'title': 'Machine Learning Engineering in Action',
   'file_path': 'data\\pdfs\\Ben Wilson - Machin

In [15]:


rag_retriever.retrieve("What is Reed frog (Hyperolius spinigularis) tadpole mortality?")

Retrieving documents for query: 'What is Reed frog (Hyperolius spinigularis) tadpole mortality?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.96it/s]

Retrieved 5 documents (after filtering)


[{'id': 'doc_f7c81868_38',
  'content': 'olius spinigularis) tadpole mortality.156 The natural history background to these data is very\ninteresting. Take a look at the full paper, if amphibian life history dynamics interests you.',
  'metadata': {'author': '',
   'filename': 'RM-StatRethink-Bayes.pdf',
   'filetype': 'pdf',
   'title': '',
   'creationDate': "D:20151109142400+01'00'",
   'modDate': "D:20190726081752-06'00'",
   'moddate': '2019-07-26T08:17:52-06:00',
   'page': 370,
   'file_path': 'data\\pdfs\\RM-StatRethink-Bayes.pdf',
   'creationdate': '2015-11-09T14:24:00+01:00',
   'keywords': '',
   'format': 'PDF 1.5',
   'creator': 'LaTeX with hyperref package',
   'doc_index': 38,
   'subject': '',
   'source': 'data\\pdfs\\RM-StatRethink-Bayes.pdf',
   'content_length': 188,
   'producer': 'XeTeX 0.99992',
   'total_pages': 483,
   'trapped': ''},
  'similarity_score': 0.6824672222137451,
  'distance': 0.3175327777862549,
  'rank': 1},
 {'id': 'doc_2f8023d6_38',
  'content'

In [16]:
#  # Simple RAG pipeline with LLama3 LLM

# from langchain_ollama import ChatOllama
# import os
# from dotenv import load_dotenv
# load_dotenv()

# ## Open AI LLM call
# llm = ChatOllama(model="llama3")

# # Simple RAG function: retrieve context + generate response
# def rag_simple(query, retriever, llm, top_k=3):
#     # Retrieve the context
#     results=retriever.retrieve(query, top_k=top_k)
#     context= '\n\n'.join([doc['content'] for doc in results]) if results else ""
#     if not context:
#         return "No relevant context found to answer the question."

#     # Generate the answer using Llama3 LLM
#     prompt = f""" Use the following context to answer the question concisely.
#             Context: {context}
#             Question: {query}
#             Answer:"""

#     response=llm.invoke([prompt.format(context=context, query=query)])

#     return response.content

In [17]:
# answer = rag_simple("What is varying intercepts model?", rag_retriever, llm)
# print(answer)

In [18]:
# from langchain_ollama import OllamaLLM
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# # 1. Use the more efficient LLM class
# # temperature=0 ensures faster, more deterministic factual answers
# # llm = OllamaLLM(model="llama3", temperature=0)

# llm = OllamaLLM(
#     model="llama3", 
#     temperature=0,
#     num_ctx=4096  # <--- THIS IS THE KEY
# )

# def rag_optimized(query, retriever, top_k=3):
#     # 2. Retrieve only what is necessary
#     results = retriever.retrieve(query, top_k=top_k)
#     context = '\n\n'.join([doc['content'] for doc in results]) if results else ""
    
#     if not context:
#         return "No relevant context found."

#     # 3. Use a formal Prompt Template (Faster serialization than f-strings)
#     template = """Use the following context to answer the question concisely.
#     Context: {context}
#     Question: {query}
#     Answer:"""
    
#     prompt = ChatPromptTemplate.from_template(template)

#     # 4. Create an LCEL Chain
#     # This pre-compiles the logic for faster execution
#     chain = prompt | llm | StrOutputParser()

#     # 5. Invoke (or use .stream() for real-time output)
#     return chain.invoke({"context": context, "query": query})

In [19]:
# answer = rag_optimized("What is varying intercepts model?", rag_retriever)
# print(answer)

In [20]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Use OllamaLLM (lighter than ChatOllama) with tuned parameters
# llm = OllamaLLM(
#     model="llama3",
#     temperature=0,
#     num_ctx=2048,      # Small window = Fast response
#     num_thread=8       # Adjust to your CPU core count
# )

# llm = OllamaLLM(
#     model="llama3.2",
#     temperature=0,
#     # PERFORMANCE TWEAKS:
#     num_ctx=2048,           # Smaller memory footprint
#     keep_alive=-1,          # Prevent reloading every 5 mins
#     num_predict=256,        # Cap the response length to stay concise
#     num_gpu=1,              # Ensure it's using the main GPU
#     repeat_penalty=1.1      # Minor quality boost without speed cost
# )

llm = OllamaLLM(
    model="llama3.2",
    temperature=0,
    # PERFORMANCE TWEAKS:
    num_ctx=4096,           # Increased from 2048! CUDA 13 handles this easily on 8GB VRAM
    keep_alive=-1,          # Keeps the model in VRAM indefinitely
    num_predict=256,        
    # CRITICAL CHANGE: 
    # Use a high number to ensure ALL layers are offloaded to your 1070 Max-Q
    num_gpu=35,             
    repeat_penalty=1.1      
)

def rag_fast(query, retriever, top_k=3):
    # Retrieve only 3 chunks to keep the prompt small
    results = retriever.retrieve(query, top_k=top_k)
    context = '\n\n'.join([doc['content'] for doc in results])
    
    # Use LCEL for minimal Python overhead
    template = "Context: {context}\n\nQuestion: {query}\n\nAnswer concisely:"
    prompt = ChatPromptTemplate.from_template(template)
    
    chain = prompt | llm | StrOutputParser()
    
    # Use .stream() if you want to see the answer as it generates!
    return chain.invoke({"context": context, "query": query})

In [22]:
answer = rag_fast("What is continuous mixture model?", rag_retriever)
print(answer)


Retrieving documents for query: 'What is continuous mixture model?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.82it/s]

Retrieved 3 documents (after filtering)


A continuous mixture model is a statistical distribution that represents a combination of multiple underlying distributions, where each component has its own probability distribution (e.g., beta-distributed probabilities).


In [23]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.7, return_context=False):
    """
    RAG pipeline with extra features:
    -Return answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return{'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    #Prepare context and sources
    context = '\n\n'.join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        # 'preview': doc['content'] + "..."
        'preview': doc['content'][:150].rsplit(' ', 1)[0] + "..."
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f""" 
            Use the following context to answer the question concisely.
            Context: {context}
            Question:{query}
            Answer:    
            """
    response = llm.invoke(
        [prompt.format(context=context, query=query)]
    )

    output = {
        'answer': response if isinstance(response, str) else response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    
    return output

result = rag_advance('What is varying continuous mixture model?', rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'])

Retrieving documents for query: 'What is varying continuous mixture model?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 docuemnts


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.59it/s]

Retrieved 3 documents (after filtering)


Answer: A varying continuous mixture model is a type of distribution where each component in the mixture has its own independent beta-distributed probability, and these components are assumed to have different parameters that vary across observations.
Sources: [{'source': 'data\\pdfs\\RM-StatRethink-Bayes.pdf', 'page': 364, 'score': 0.6166026592254639, 'preview': 'Overthinking: Continuous mixtures. A distribution like the beta-binomial is called a continuous\nmixture, because every binomial count is assumed to...'}, {'source': 'data\\pdfs\\RM-StatRethink-Bayes.pdf', 'page': 364, 'score': 0.6166026592254639, 'preview': 'Overthinking: Continuous mixtures. A distribution like the beta-binomial is called a continuous\nmixture, because every binomial count is assumed to...'}, {'source': 'data\\pdfs\\RM-StatRethink-Bayes.pdf', 'page': 355, 'score': 0.5272160768508911, 'preview': 'they are mixtures of multiple processes. Whenever there are different causes for the same\nobservation, then a mi

In [26]:
# from ragas.testset.generator import TestsetGenerator
from ragas.testset import TestsetGenerator
# from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI # Or your Ollama wrapper

from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

# 1. Initialize your synthesizers (The new "Evolutions")
# You pass your generator_llm to these directly



# 1. Load your PDFs (e.g., Statistical Rethinking)
loader = PyPDFLoader("D:\DataScienceProjects\Langchain-RAG-Optimized\data\pdfs\RM-StatRethink-Bayes.pdf")
documents = loader.load()

# 2. Configure your "Teacher" Models
# Tip: Use your Ollama Llama 3.1 8B here for higher quality questions
generator_llm = ChatOpenAI(model="llama3.1", base_url="http://localhost:11434/v1", api_key="ollama")
critic_llm = ChatOpenAI(model="llama3.1", base_url="http://localhost:11434/v1", api_key="ollama")

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25)
]

generator = TestsetGenerator(
    generator_llm,
    critic_llm,
    embeddings=embedding_manager #my_local_embedding_model
)

# 3. Generate the "Golden" Dataset
# distributions determine the mix of easy vs. hard questions
testset = generator.generate(
    documents=documents,
    testset_size=20,
    query_distribution=query_distribution
)
# 4. Save to CSV for your Git repo
test_df = testset.to_pandas()
test_df.to_csv("data/golden_test_set.csv", index=False)

TypeError: TestsetGenerator.__init__() got an unexpected keyword argument 'embeddings'

In [ ]:
import os
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader

# 1. Setup environment to bypass OpenAI key checks
os.environ["OPENAI_API_KEY"] = "ollama"

# 2. Initialize your local LLM (Ollama)
# Using the same instance for generator and critic saves VRAM on your 1070
local_model = ChatOpenAI(
    model="llama3.2:latest", 
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
generator_llm = LangchainLLMWrapper(local_model)

# 3. Initialize your EmbeddingManager (Assuming it inherits from LangChain's Embeddings)
# Use the wrapper to make it Ragas-compatible
generator_embeddings = LangchainEmbeddingsWrapper(embedding_manager)

# 4. Define the Query Distribution (The new "Evolutions")
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25)
]

# 5. Initialize the Generator
generator = TestsetGenerator(
    llm=generator_llm, 
    embedding_model=generator_embeddings
)

# 6. Load your Borges PDF (Use Absolute Path to avoid FileNotFoundError)
path = os.path.abspath("D:\DataScienceProjects\Langchain-RAG-Optimized\data\pdfs\RM-StatRethink-Bayes.pdf")
loader = PyPDFLoader(path)
docs = loader.load()

# 7. Generate!
# We set testset_size=10 as requested
testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=10,
    query_distribution=query_distribution
)

# 8. Export to CSV for your project
test_df = testset.to_pandas()
test_df.to_csv("data/golden_test_set.csv", index=False)
print("✅ Test set generated and saved to data/golden_test_set.csv")

C:\Users\Sumit Shrestha\AppData\Local\Temp\ipykernel_21536\200838135.py:23: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(local_model)
C:\Users\Sumit Shrestha\AppData\Local\Temp\ipykernel_21536\200838135.py:27: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embedding_manager)
Applying HeadlinesExtractor:  13%|█▎        | 55/426 [04:07<23:04,  3.73s/it]  

In [29]:
from google.cloud import storage


def list_arxiv_pdfs(bucket_name, prefix):
    storage_client = storage.Client.create_anonymous_client()
    blobs = storage_client.list_blobs(bucket_name, prefix=prefix)
    
    for blob in blobs:
        if blob.name.endswith('.pdf'):
            print(f"Found: {blob.name}")

# List PDFs from January 2024
list_arxiv_pdfs("arxiv-dataset", "arxiv/arxiv/pdf/2401/")

Found: arxiv/arxiv/pdf/2401/2401.00001v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00002v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v2.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v3.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v4.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v5.pdf
Found: arxiv/arxiv/pdf/2401/2401.00003v6.pdf
Found: arxiv/arxiv/pdf/2401/2401.00004v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00005v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00006v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00006v2.pdf
Found: arxiv/arxiv/pdf/2401/2401.00006v3.pdf
Found: arxiv/arxiv/pdf/2401/2401.00007v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00008v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00009v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00009v2.pdf
Found: arxiv/arxiv/pdf/2401/2401.00009v3.pdf
Found: arxiv/arxiv/pdf/2401/2401.00010v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00011v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00012v1.pdf
Found: arxiv/arxiv/pdf/2401/2401.00013v1.pdf
Found: arx